In [ ]:
!pip install opencv-python torch torchvision matplotlib scikit-learn gradio -q

In [ ]:
import os
import cv2
import torch
import shutil
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader, random_split
from sklearn.cluster import KMeans
from PIL import Image
import gradio as gr
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from torchvision.datasets import Places365

# Download validation set (small enough for Colab)
val_dataset = Places365(root='./places365', split='val', small=True, download=True)
print(f"Dataset downloaded. Number of images: {len(val_dataset)}")

Dataset downloaded. Number of images: 36500


In [ ]:
# The dataset has a 'classes' attribute with 365 scene names
categories = val_dataset.classes
print(f"First 5 categories: {categories[:5]}")

# Map scene categories to your 4 styles
style_map = {}
for cat in categories:
    if cat in ['art_studio', 'loft', 'home_office', 'conference_room', 'office', 'kindergarten', 'gym']:
        style_map[cat] = 'Modern'
    elif cat in ['closet', 'pantry', 'bathroom', 'storage_room', 'utility_room', 'locker_room']:
        style_map[cat] = 'Minimalist'
    elif cat in ['cabin', 'cottage', 'farm', 'hunting_lodge', 'basement', 'garage', 'barn']:
        style_map[cat] = 'Rustic'
    elif cat in ['ballroom', 'dining_room', 'library', 'mansion', 'church', 'museum', 'restaurant', 'concert_hall']:
        style_map[cat] = 'Classic'

print(f"Mapped {len(style_map)} categories to styles.")

First 5 categories: ['/a/airfield', '/a/airplane_cabin', '/a/airport_terminal', '/a/alcove', '/a/alley']
Mapped 0 categories to styles.


In [ ]:
import os
import shutil
from torchvision.datasets import Places365

# Reload dataset if needed
if 'val_dataset' not in locals():
    val_dataset = Places365(root='./places365', split='val', small=True, download=True)

categories = val_dataset.classes
print(f"Total categories: {len(categories)}")

# Function to clean category name (remove '/a/' prefix)
def clean_cat(cat):
    return cat.replace('/a/', '')

# Create destination folders
dest_dir = './places365_styled'
for style in ['Modern', 'Minimalist', 'Rustic', 'Classic']:
    os.makedirs(os.path.join(dest_dir, style), exist_ok=True)

# Define style keywords (match after cleaning)
style_keywords = {
    'Modern': ['loft', 'studio', 'office', 'conference_room', 'kindergarten', 'gym'],
    'Minimalist': ['closet', 'pantry', 'storage', 'utility', 'locker', 'bathroom'],
    'Rustic': ['cabin', 'cottage', 'farm', 'lodge', 'basement', 'garage', 'barn'],
    'Classic': ['ballroom', 'dining', 'library', 'mansion', 'church', 'museum', 'restaurant', 'concert', 'living_room', 'bedroom']
}

# Build mapping from cleaned category name to style
style_map = {}
for idx, cat in enumerate(categories):
    clean = clean_cat(cat)
    for style, keywords in style_keywords.items():
        if any(kw in clean for kw in keywords):
            style_map[cat] = style
            break

print(f"Mapped {len(style_map)} categories to styles.")
print("Example mapped categories:", list(style_map.items())[:10])

# Copy images
copied_count = 0
total_processed = 0
for img_path, label_idx in val_dataset.imgs:
    total_processed += 1
    cat_name = categories[label_idx]
    if cat_name in style_map:
        style = style_map[cat_name]
        fname = os.path.basename(img_path)
        dest_path = os.path.join(dest_dir, style, fname)
        shutil.copy(img_path, dest_path)
        copied_count += 1
        if copied_count % 100 == 0:
            print(f"Copied {copied_count} images...")
    if total_processed % 2000 == 0:
        print(f"Processed {total_processed} images...")

print(f"Total images copied: {copied_count}")

# Verify
for style in ['Modern', 'Minimalist', 'Rustic', 'Classic']:
    style_path = os.path.join(dest_dir, style)
    num = len([f for f in os.listdir(style_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"{style}: {num} images")

Total categories: 365
Mapped 50 categories to styles.
Example mapped categories: [('/a/airplane_cabin', 'Rustic'), ('/a/art_studio', 'Modern'), ('/a/artists_loft', 'Modern'), ('/b/ballroom', 'Classic'), ('/b/barn', 'Rustic'), ('/b/barndoor', 'Rustic'), ('/b/basement', 'Rustic'), ('/b/bathroom', 'Minimalist'), ('/b/bedroom', 'Classic'), ('/c/cabin/outdoor', 'Rustic')]
Copied 100 images...
Copied 200 images...
Processed 2000 images...
Copied 300 images...
Copied 400 images...
Copied 500 images...
Processed 4000 images...
Copied 600 images...
Copied 700 images...
Copied 800 images...
Processed 6000 images...
Copied 900 images...
Copied 1000 images...
Copied 1100 images...
Processed 8000 images...
Copied 1200 images...
Copied 1300 images...
Copied 1400 images...
Processed 10000 images...
Copied 1500 images...
Copied 1600 images...
Processed 12000 images...
Copied 1700 images...
Copied 1800 images...
Copied 1900 images...
Processed 14000 images...
Copied 2000 images...
Copied 2100 images...

In [ ]:
from torchvision import datasets

# Check if destination folder exists and has images
if os.path.exists(dest_dir):
    styled_dataset = datasets.ImageFolder(root=dest_dir)
    print("Classes found:", styled_dataset.classes)
    print("Number of images per class:")
    for c in styled_dataset.classes:
        class_path = os.path.join(dest_dir, c)
        num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {c}: {num_images} images")
    print(f"Total images in dataset: {len(styled_dataset)}")
else:
    print(f"Error: {dest_dir} does not exist. Run Cell 5 first.")

Classes found: ['Classic', 'Minimalist', 'Modern', 'Rustic']
Number of images per class:
  Classic: 1800 images
  Minimalist: 600 images
  Modern: 1200 images
  Rustic: 1400 images
Total images in dataset: 5000


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Set transform for the full dataset
styled_dataset.transform = train_transform

# Split 80/20
train_size = int(0.8 * len(styled_dataset))
val_size = len(styled_dataset) - train_size
train_dataset, val_dataset_split = random_split(styled_dataset, [train_size, val_size])

# Use different transform for validation
val_dataset_split.dataset.transform = val_transform

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset_split, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 125, Val batches: 32


In [ ]:
num_classes = len(styled_dataset.classes)  # should be 4
model = models.mobilenet_v2(pretrained=True)

# Freeze feature extractor (train only classifier for speed)
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

num_epochs = 15
best_val_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_style_model.pth')

print(f"Training finished. Best validation accuracy: {best_val_acc:.2f}%")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/15, Loss: 0.8998, Val Acc: 65.40%
Epoch 2/15, Loss: 0.7689, Val Acc: 69.90%
Epoch 3/15, Loss: 0.7400, Val Acc: 70.20%
Epoch 4/15, Loss: 0.7008, Val Acc: 68.10%
Epoch 5/15, Loss: 0.7019, Val Acc: 71.90%
Epoch 6/15, Loss: 0.6550, Val Acc: 71.40%
Epoch 7/15, Loss: 0.6504, Val Acc: 72.10%
Epoch 8/15, Loss: 0.5961, Val Acc: 72.90%
Epoch 9/15, Loss: 0.5792, Val Acc: 72.40%
Epoch 10/15, Loss: 0.5554, Val Acc: 73.20%
Epoch 11/15, Loss: 0.5310, Val Acc: 70.60%
Epoch 12/15, Loss: 0.5033, Val Acc: 73.30%
Epoch 13/15, Loss: 0.4990, Val Acc: 72.00%
Epoch 14/15, Loss: 0.4873, Val Acc: 73.50%
Epoch 15/15, Loss: 0.4644, Val Acc: 74.70%
Training finished. Best validation accuracy: 74.70%


In [ ]:
model.load_state_dict(torch.load('best_style_model.pth', map_location=device))
model.eval()
class_names = styled_dataset.classes   # e.g. ['Classic','Minimalist','Modern','Rustic']

def predict_style(image_path):
    """Predict style from image file path."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)
    return class_names[predicted.item()]

In [ ]:
def extract_palette(image_path, k=5):
    """Return list of HEX codes for k dominant colors."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not read image")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pixels = img.reshape(-1, 3).astype(np.float32)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels)
    centers = kmeans.cluster_centers_.astype(int)
    palette_hex = [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in centers]
    return palette_hex

In [ ]:
recommendations = {
    "Modern": {
        "furniture": "Clean lines, metal/glass surfaces, neutral upholstery",
        "layout": "Open plan, minimal clutter, statement artwork",
        "lighting": "Track lighting, large windows, LED strips"
    },
    "Minimalist": {
        "furniture": "Functional pieces, hidden storage, monochrome tones",
        "layout": "Plenty of negative space, simple geometric shapes",
        "lighting": "Recessed lights, natural light priority"
    },
    "Rustic": {
        "furniture": "Reclaimed wood, leather sofa, wrought iron details",
        "layout": "Cozy arrangement, central fireplace or focal point",
        "lighting": "Warm bulbs, lanterns, wrought iron chandeliers"
    },
    "Classic": {
        "furniture": "Tufted sofas, dark wood, elegant details",
        "layout": "Symmetrical, formal seating areas",
        "lighting": "Crystal chandeliers, sconces"
    }
}

def get_recommendations(style, palette_hex):
    base = recommendations.get(style, recommendations["Modern"])
    # (Optional) refine based on palette – keep simple for capstone
    return base

In [ ]:
# ==============================================
# FINAL GRADIO CELL – Color boxes working
# ==============================================

import gradio as gr
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(pretrained=False)
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(0.2),
    torch.nn.Linear(model.last_channel, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 4)
)
model.load_state_dict(torch.load('best_style_model.pth', map_location=device))
model.eval()
model.to(device)

class_names = ['Classic', 'Minimalist', 'Modern', 'Rustic']

def predict_style(image):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    img_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(img_tensor)
        pred = torch.argmax(out, 1).item()
    return class_names[pred]

def extract_palette(image, k=5):
    img = np.array(image.convert('RGB'))
    pixels = img.reshape(-1, 3).astype(np.float32)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels)
    centers = kmeans.cluster_centers_.astype(int)
    return [f"#{r:02x}{g:02x}{b:02x}" for r,g,b in centers]

recommendations = {
    "Modern": {
        "furniture": "Clean lines, metal/glass surfaces, neutral upholstery",
        "layout": "Open plan, minimal clutter, statement artwork",
        "lighting": "Track lighting, large windows, LED strips"
    },
    "Minimalist": {
        "furniture": "Functional pieces, hidden storage, monochrome tones",
        "layout": "Plenty of negative space, simple shapes",
        "lighting": "Recessed lights, natural light priority"
    },
    "Rustic": {
        "furniture": "Reclaimed wood, leather sofa, wrought iron details",
        "layout": "Cozy, central fireplace or focal point",
        "lighting": "Warm bulbs, lanterns, wrought iron chandeliers"
    },
    "Classic": {
        "furniture": "Tufted sofas, dark wood, elegant details",
        "layout": "Symmetrical, formal seating areas",
        "lighting": "Crystal chandeliers, sconces"
    }
}

def interior_advisor(image):
    if image is None:
        return "<div style='padding:20px; text-align:center; color: #333;'>⚠️ Please upload an image of a room.</div>"

    style = predict_style(image)
    palette = extract_palette(image)
    rec = recommendations[style]

    # Create color boxes with explicit text color
    swatches_html = ""
    for hexc in palette:
        swatches_html += f"""
        <div style="display: inline-block; margin: 8px; text-align: center;">
            <div style="background-color: {hexc}; width: 80px; height: 80px; border-radius: 12px; border: 2px solid #ddd; box-shadow: 0 2px 5px rgba(0,0,0,0.2);"></div>
            <div style="font-family: monospace; margin-top: 5px; color: #1e1e2f;">{hexc}</div>
        </div>
        """

    html_output = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; color: #1e1e2f; background: #fefefe; padding: 20px; border-radius: 16px; margin: 10px 0;">
        <h2 style="color: #1e1e2f;">🎨 Detected Style: <span style="color: #2c3e50;">{style}</span></h2>

        <h3 style="color: #1e1e2f;">🖌️ Color Palette</h3>
        <div style="margin-bottom: 20px;">{swatches_html}</div>

        <div style="background: #f0f2f5; padding: 15px; border-radius: 12px; margin-top: 15px; color: #1e1e2f;">
            <h3 style="color: #1e1e2f;">🛋️ Furniture Suggestions</h3>
            <p style="color: #1e1e2f;">{rec['furniture']}</p>

            <h3 style="color: #1e1e2f;">📐 Layout Advice</h3>
            <p style="color: #1e1e2f;">{rec['layout']}</p>

            <h3 style="color: #1e1e2f;">💡 Lighting Tips</h3>
            <p style="color: #1e1e2f;">{rec['lighting']}</p>
        </div>
    </div>
    """
    return html_output

# Launch interface
iface = gr.Interface(
    fn=interior_advisor,
    inputs=gr.Image(type="pil", label="📸 Upload a Room Photo"),
    outputs=gr.HTML(label="✨ Design Recommendations"),  # HTML output, not Markdown
    title="🏡 Interior Design Advisor",
    description="Upload a photo of a room – AI will predict the design style, extract the dominant colors, and give you personalized furniture, layout, and lighting advice."
)

iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6d5836b2f339969ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
